<a href="https://colab.research.google.com/github/yaseen141199/Branches/blob/main/Copy_of_Untitled7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══════════════════════════════════════════
VIDEO_URL    = 'https://www.youtube.com/playlist?list=PLZmPGUyBFvUqo76bXGnXq9EofsaV2d8K5'
QUALITY      = '480'      # 360 | 480 | 720 | 1080 | 1440 | '' (فاضي = اقصى جودة متاحة)
START        = '0'        # لو الرابط فيديو وعايز تقص منه هتكتب هنا زمن بداية القص  | لو قائمة تشغيل:هتكتب هنا رقم اول فيديو
END          = '0'        #  لو الرابط فيديو وعايز تقص منه هتكتب هنا زمن نهاية القص و-1 يعني لاخر الفيديو  | لو قائمة تشغيل:هتكتب هنا رقم اخر فيديو و-1 يعني لاخر فيديو في القائمة
SUBTITLES    = False      # True = تنزيل الترجمات
SUB_LANGS    = 'en,ar'    # فاضي = كل الترجمات المتاحة | او تكتب كود اللغة:
AUDIO_ONLY   = False      # True = تنزيل الصوت فقط | False = تنزيل الفيديو
AUDIO_FMT    = 'mp3'      # mp3 | wav | aac | flac | ogg | m4a صيغة الصوت
# ═══════════════════════════════════════════

"""
ده كود بينزّلك فيديو من يوتيوب، او قائمة تشغيل كاملة
تقدر تنزّل الفيديو زي ما هو، او تنزّل بس الصوت، وتقدر كمان تقص جزء معين
 من الفيديو/الصوت (يعني تحدد من ثانية كام لحد ثانية كام)، وتنزّل
الترجمة معاه لو موجودة. ولو بتنزّل بلاي ليست وفيها فيديو محذوف او مخفي،
الكود هيتخطاه وهيكمل ينزّل باقي الفيديوهات عادي.

دلوقتي هنشرح كل سطر تحت بيعمل ايه وتقدر تغيره بايه:

- VIDEO_URL: حط هنا لينك الفيديو، او لينك البلايليست لو هتنزل قائمة كاملة.

- الجودة
- QUALITY:  اختار واحدة من360 | 480 | 720 | 1080 | 1440
:
  لو سيبتها فاضية '' هيجيبلك اعلى جودة موجودة

- START: من امتى تبدأ.
    - لو فيديو واحد: حط زمن البداية، مثلا '0' يعني من الاول، او '90' (يعني 90 ثانية)، او '1:30'، او '1h30m'.
    - لو بلايليست: حط رقم اول فيديو عايز تبدأ منه (يعني لو كتبت 1 يبدأ من اول فيديو في القائمة).
    - لو سبتها اصفار زي م هي هينزل القائمة كلها

- END: لحد امتى تخلص.
    - لو فيديو هتحط اخر وقت عايز يقص فيه ولو عملت -1 كده هيقص من الزمن اللي كتبته في البداية الي نهاية الفيديو
    - لو بلايليست: حط رقم اخر فيديو عايز تقف عنده. لو حطيت '-1' معناها هينزل لحد اخر فيديو في القائمة.

- AUDIO_ONLY: لو عايز تنزل الصوت بس (بدون فيديو) حطها ترو ولو عايز الفيديو حطها فولس.
  لو حطيتها ترو القص بتاع البداية و النهاية هيتطبق على الصوت برضو.

- AUDIO_FMT: صيغة الصوت اللي عايزها لو AUDIO_ONLY=True. اختار من: mp3 | wav | aac | flac | ogg | m4a

- SUBTITLES: لو عايز تنزل الترجمة مع الفيديو حطها ترو لو لا حطها فولس.

- SUB_LANGS: لو عايز ترجمة بلغة معينة حط رمزها هنا.
  لو سيبتها فاضية '' هينزل كل الترجمات الموجودة.
"""

import subprocess, re, os, json, shutil, sys
from google.colab import files

subprocess.run(['pip', 'install', '-q', '--upgrade', 'yt-dlp'], check=True, capture_output=True)

WORKDIR = '/content/_yt_work'
shutil.rmtree(WORKDIR, ignore_errors=True)
os.makedirs(WORKDIR, exist_ok=True)

QUALITY_ORDER = ['1440', '1080', '720', '480', '360']
AUDIO_CODEC_MAP = {
    'mp3':  ['-vn', '-acodec', 'libmp3lame', '-q:a', '2'],
    'wav':  ['-vn', '-acodec', 'pcm_s16le'],
    'aac':  ['-vn', '-acodec', 'aac', '-b:a', '192k'],
    'flac': ['-vn', '-acodec', 'flac'],
    'ogg':  ['-vn', '-acodec', 'libvorbis', '-q:a', '4'],
    'm4a':  ['-vn', '-acodec', 'aac', '-b:a', '192k'],
}

def fatal(msg):
    print(f'❌ خطأ: {msg}')
    sys.exit(1)

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', '_', name).strip()


def parse_time(raw):
    raw = str(raw).strip()
    if re.match(r'^\d{1,2}:\d{2}(:\d{2})?$', raw):
        p = raw.split(':')
        if len(p) == 2:
            t = int(p[0]) * 60 + int(p[1])
        else:
            t = int(p[0]) * 3600 + int(p[1]) * 60 + int(p[2])
    elif re.search(r'[hHmMsS]', raw):
        h = int(re.search(r'(\d+)[hH]', raw).group(1)) if re.search(r'(\d+)[hH]', raw) else 0
        m = int(re.search(r'(\d+)[mM]', raw).group(1)) if re.search(r'(\d+)[mM]', raw) else 0
        s = int(re.search(r'(\d+)[sS]', raw).group(1)) if re.search(r'(\d+)[sS]', raw) else 0
        t = h * 3600 + m * 60 + s
    else:
        n = float(raw)
        t = int(n * 60) if n < 100 else int(n)
    h2, r = divmod(t, 3600)
    m2, s2 = divmod(r, 60)
    return f'{h2:02d}:{m2:02d}:{s2:02d}'


def to_sec(t):
    h, m, s = map(int, t.split(':'))
    return h * 3600 + m * 60 + s


def run_quiet(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)


def get_flat_info(url):
    r = run_quiet(['yt-dlp', '-J', '--flat-playlist', url])
    if r.returncode != 0:
        fatal(f'تعذر جلب معلومات الرابط:\n{r.stderr}')
    return json.loads(r.stdout)


def get_full_info(url):
    r = run_quiet(['yt-dlp', '-J', '--no-playlist', url])
    if r.returncode != 0:
        print(f'❌ خطأ: تعذر جلب معلومات الفيديو ({url}):\n{r.stderr}')
        return None
    try:
        return json.loads(r.stdout)
    except Exception:
        print(f'❌ خطأ: تعذر تحليل بيانات الفيديو ({url})')
        return None


def short_lang(code):
    return str(code).split('-')[0].split('_')[0].lower().strip()


def filter_sub_langs(requested_csv, available_codes):
    if not available_codes:
        return []
    if not requested_csv or not str(requested_csv).strip():
        return list(available_codes)
    requested = [x.strip() for x in str(requested_csv).split(',') if x.strip()]
    result = []
    for req in requested:
        req_s = short_lang(req)
        if req in available_codes:
            if req not in result:
                result.append(req)
            continue
        for code in available_codes:
            if short_lang(code) == req_s:
                if code not in result:
                    result.append(code)
    return result


def quality_format_chain(requested_quality):
    if not requested_quality or not str(requested_quality).strip():
        return 'bestvideo+bestaudio/best'
    start_idx = QUALITY_ORDER.index(requested_quality)
    chain_parts = [f'bestvideo[height<={q}]+bestaudio' for q in QUALITY_ORDER[start_idx:]]
    chain_parts.append('bestvideo+bestaudio')
    chain_parts.append('best')
    return '/'.join(chain_parts)


def get_video_duration_sec(video_file):
    r = run_quiet([
        'ffprobe', '-v', 'error',
        '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1',
        video_file
    ])
    if r.returncode == 0:
        try:
            return float(r.stdout.strip())
        except Exception:
            pass
    return None


def sec_to_hms(total_sec):
    total_sec = int(total_sec)
    h, r = divmod(total_sec, 3600)
    m, s = divmod(r, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'

def validate_inputs():
    q = str(QUALITY).strip()
    if q and q not in QUALITY_ORDER:
        fatal(f'قيمة الجودة "{QUALITY}" غير معروفة. الخيارات: {QUALITY_ORDER} او فارغة لاقصى جودة.')

    fmt = str(AUDIO_FMT).strip().lower()
    if fmt not in AUDIO_CODEC_MAP:
        fatal(f'صيغة الصوت "{AUDIO_FMT}" غير مدعومة. الخيارات: {list(AUDIO_CODEC_MAP.keys())}')

    s = str(START).strip()
    e = str(END).strip()
    if not s:
        fatal('قيمة البداية (START) فارغة، اكتب 0 على الاقل.')
    if not e:
        fatal('قيمة النهاية (END) فارغة، اكتب 0 على الاقل.')


def build_video_list(url):
    """يرجع (video_items, is_playlist)."""
    info = get_flat_info(url)
    is_playlist = info.get('_type') == 'playlist' or 'entries' in info

    if not is_playlist:
        return [{'url': url}], False

    entries = [e for e in info.get('entries', []) if e]
    total = len(entries)
    if total == 0:
        fatal('قائمة التشغيل فارغة او تعذر قراءتها.')

    start_s = str(START).strip()
    end_s = str(END).strip()

    start_idx = int(start_s) if start_s not in ('0', '') else 1
    if start_idx < 1:
        start_idx = 1

    if end_s == '-1':
        end_idx = total
    else:
        end_idx = int(end_s) if end_s != '0' else total

    if end_idx > total:
        end_idx = total
    if start_idx > end_idx:
        fatal(f'رقم البداية ({start_idx}) اكبر من رقم النهاية ({end_idx}) في قائمة التشغيل (اجمالي {total} فيديو).')

    selected = entries[start_idx - 1:end_idx]
    items = []
    for e in selected:
        vid_id = e.get('id')
        vid_url = e.get('url') or f'https://www.youtube.com/watch?v={vid_id}'
        items.append({'url': vid_url})
    return items, True


def download_one(item, idx, is_playlist):
    url = item['url']
    full_info = get_full_info(url)
    if full_info is None:
        return None

    raw_title = full_info.get('title') or f'video_{idx}'
    title = sanitize_filename(raw_title)
    job_dir = os.path.join(WORKDIR, f'job_{idx}')
    os.makedirs(job_dir, exist_ok=True)
    base_path = os.path.join(job_dir, title)

    no_cut = True
    start_str = end_str = None
    use_end_of_file = False

    if not is_playlist:
        start_s = str(START).strip()
        end_s = str(END).strip()
        no_cut = (start_s == '0' and end_s == '0')

        if not no_cut:
            if end_s == '-1':
                use_end_of_file = True
                try:
                    start_str = parse_time(START)
                except Exception:
                    print(f'❌ خطأ: قيمة البداية غير صالحة: "{START}" — تخطي القص.')
                    no_cut = True
                    use_end_of_file = False
            else:
                try:
                    start_str = parse_time(START)
                    end_str = parse_time(END)
                except Exception:
                    print(f'❌ خطأ: قيمة بداية/نهاية غير صالحة: "{START}" / "{END}" — تخطي القص.')
                    no_cut = True
                else:
                    if to_sec(end_str) <= to_sec(start_str):
                        print(f'❌ خطأ: وقت النهاية ({end_str}) قبل او يساوي وقت البداية ({start_str}) — تخطي القص.')
                        no_cut = True

    video_file = None

    if AUDIO_ONLY:
        out_tmpl = base_path + '.%(ext)s'
        r = run_quiet(['yt-dlp', '-f', 'bestaudio/best', '-o', out_tmpl, '--no-playlist', url])
        if r.returncode != 0:
            print(f'❌ خطأ اثناء تنزيل الصوت ({title}):\n{r.stderr}')
            shutil.rmtree(job_dir, ignore_errors=True)
            return None
        for f in os.listdir(job_dir):
            fp = os.path.join(job_dir, f)
            if os.path.isfile(fp) and f.startswith(os.path.basename(base_path) + '.'):
                video_file = fp
                break

    else:
        fmt_selector = quality_format_chain(QUALITY)
        full_path = base_path + '.mp4'
        r = run_quiet(['yt-dlp', '-f', fmt_selector, '--merge-output-format', 'mp4',
                       '-o', full_path, '--no-playlist', url])
        if r.returncode != 0:
            print(f'❌ خطأ اثناء تنزيل الفيديو ({title}):\n{r.stderr}')
            shutil.rmtree(job_dir, ignore_errors=True)
            return None
        video_file = full_path

    if video_file is None or not os.path.exists(video_file):
        print(f'❌ خطأ: لم يتم العثور على ملف الفيديو بعد التنزيل ({title})')
        shutil.rmtree(job_dir, ignore_errors=True)
        return None

    if use_end_of_file and start_str is not None:
        dur = get_video_duration_sec(video_file)
        if dur is not None:
            end_str = sec_to_hms(dur)
            if to_sec(end_str) <= to_sec(start_str):
                print(f'❌ خطأ: وقت البداية ({start_str}) بعد نهاية الفيديو ({end_str}) — تخطي القص.')
                no_cut = True
        else:
            print(f'❌ خطأ: تعذر تحديد مدة الفيديو — تخطي القص.')
            no_cut = True

    sub_files = []
    if SUBTITLES:
        available_subs = full_info.get('subtitles') or {}
        available_auto = full_info.get('automatic_captions') or {}
        all_available_codes = set(available_subs.keys()) | set(available_auto.keys())
        langs_to_get = filter_sub_langs(SUB_LANGS, all_available_codes)
        if langs_to_get:
            sub_tmpl = base_path + '.%(ext)s'
            run_quiet([
                'yt-dlp', '--skip-download',
                '--write-subs', '--write-auto-subs',
                '--sub-langs', ','.join(langs_to_get),
                '--sub-format', 'srt/best',
                '-o', sub_tmpl,
                '--no-playlist', url
            ])
            for f in sorted(os.listdir(job_dir)):
                fp = os.path.join(job_dir, f)
                if not os.path.isfile(fp):
                    continue
                if re.search(r'\.[a-zA-Z]{2}[a-zA-Z0-9\-]*\.(srt|vtt)$', f):
                    sub_files.append(fp)

    return {
        'title': title,
        'job_dir': job_dir,
        'video_file': video_file,
        'sub_files': sub_files,
        'no_cut': no_cut,
        'start_str': start_str,
        'end_str': end_str,
    }



def safe_download(path):
    files.download(path)


def finalize_and_download(result):
    job_dir = result['job_dir']
    title = result['title']
    video_file = result['video_file']

    if not result['no_cut']:
        ext = os.path.splitext(video_file)[1]
        cut_path = os.path.join(job_dir, f'{title} cut{ext}')
        if AUDIO_ONLY:
            # قص الصوت فقط بدون اعادة ترميز فيديو غير موجود اصلا
            cut_cmd = ['ffmpeg', '-y', '-ss', result['start_str'], '-to', result['end_str'],
                       '-i', video_file, '-c', 'copy', cut_path]
        else:
            cut_cmd = ['ffmpeg', '-y', '-ss', result['start_str'], '-to', result['end_str'],
                       '-i', video_file, '-c:v', 'libx264', '-c:a', 'aac',
                       '-preset', 'fast', '-crf', '18', cut_path]
        r = run_quiet(cut_cmd)
        if r.returncode != 0:
            print(f'❌ خطأ اثناء قص {"الصوت" if AUDIO_ONLY else "الفيديو"} ({title}):\n{r.stderr}')
        elif os.path.exists(cut_path):
            video_file = cut_path

    final_output = video_file
    if AUDIO_ONLY:
        fmt = str(AUDIO_FMT).strip().lower()
        out_path = os.path.join(job_dir, f'{title}.{fmt}')
        r = run_quiet(['ffmpeg', '-y', '-i', video_file] + AUDIO_CODEC_MAP[fmt] + [out_path])
        if r.returncode != 0:
            print(f'❌ خطأ اثناء تحويل الصوت لصيغة {fmt} ({title}):\n{r.stderr}')
        elif os.path.exists(out_path):
            final_output = out_path

    if os.path.exists(final_output):
        safe_download(final_output)
    else:
        print(f'❌ خطأ: الملف النهائي غير موجود ({title})')

    for sub_path in result['sub_files']:
        if os.path.exists(sub_path):
            safe_download(sub_path)


def main():
    validate_inputs()
    video_items, is_playlist = build_video_list(VIDEO_URL)

    for idx, item in enumerate(video_items, start=1):
        result = download_one(item, idx, is_playlist)
        if result is None:
            print(f'⚠️ تم تجاهل الفيديو رقم {idx} (غير متاح/محذوف/فشل تنزيله) — الاستمرار بباقي القائمة...')
            continue
        finalize_and_download(result)


main()

❌ خطأ اثناء تنزيل الفيديو (0- Intro - AWS Cloud Practitioner CLF02 Arabic كورس بالعربي كامل):
ERROR: unable to download video data: HTTP Error 403: Forbidden

⚠️ تم تجاهل الفيديو رقم 1 (غير متاح/محذوف/فشل تنزيله) — الاستمرار بباقي القائمة...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>